In [24]:
import os, math, datetime, random, itertools
import numpy as np
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.optim import Adam
from torch.distributions import Normal
from torch.utils.tensorboard import SummaryWriter
import gymnasium as gym
# import gymnasium_robotics
import pickle

## Utils

In [5]:
"""
对 Critic network 进行 soft update 和 hard update 的函数
Critic network 是未来的 Q 值进行预测的网络
在 DDPG 算法中，通常会有一个目标网络（target network）和一个评估网络（evaluation network）。
目标网络的参数会定期地从评估网络复制过来，这个过程可以通过软更新（soft update）或者硬更新（hard update）来实现。
"""
def soft_update(target, source, tau):
    # 使用Polyak 平均法来平滑地更新目标网络的参数
    # theta_target = tau * theta_source + (1 - tau) * theta_target
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(target_param.data * (1.0 - tau) + param.data * tau)

def hard_update(target, source):
    # 将主网络的参数直接复制到目标网络中
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(param.data)

## Model

In [ ]:
LOG_SIG_MAX = 2
LOG_SIG_MIN = -20
epsilon = 1e-6

def weights_init_(m):
    if isinstance(m, nn.Linear):
        # gain 是一个缩放因子，标识对初始化的权重数据不进行额外的放大或缩小
        # Xavier uniform 初始化方法是一种常用的权重初始化方法，适用于激活函数为 ReLU 的网络
        # gain 的设计是为了抵消不同激活函数对权重分布的影响，使得网络在训练初期能够更好地收敛
        torch.nn.init.xavier_uniform_(m.weight, gain=1)
        torch.nn.init.constant_(m.bias, 0)

class QNetwork(nn.Module):
    def __init__(self, num_inputs, num_actions, hidden_dim):
        super(QNetwork, self).__init__()

        # Q1 network 结构
        self.linear1 = nn.Linear(num_inputs + num_actions, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)
        self.linear3 = nn.Linear(hidden_dim, 1)

        # Q2 network 结构
        self.linear4 = nn.Linear(num_inputs + num_actions, hidden_dim)
        self.linear5 = nn.Linear(hidden_dim, hidden_dim)
        self.linear6 = nn.Linear(hidden_dim, 1)

        self.apply(weights_init_)
    

    def forward(self, state, action):
        xu = torch.cat([state, action], 1)

        x1 = F.relu(self.linear1(xu))
        x1 = F.relu(self.linear2(x1))
        x1 = self.linear3(x1)

        x2 = F.relu(self.linear4(xu))
        x2 = F.relu(self.linear5(x2))
        x2 = self.linear6(x2)

        return x1, x2


class GaussianPolicy(nn.Module):
    # GaussianPolicy 是 SAC 算法中使用的策略网络，输出一个高斯分布的均值和标准差，用于采样动作
    def __init__(self, num_inputs, num_actions, hidden_dim, action_space=None):
        super(GaussianPolicy, self).__init__()

        self.linear1 = nn.Linear(num_inputs, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)

        self.mean_linear = nn.Linear(hidden_dim, num_actions)
        self.log_std_linear = nn.Linear(hidden_dim, num_actions)

        self.apply(weights_init_)

        # action rescaling 动作缩放映射
        # 目的是将神经网络输处，转换为能够理解并执行的真实物理数值
        if action_space is None:
            self.action_scale = torch.tensor(1.)
            self.action_bias = torch.tensor(0.)
        else:
            self.action_scale = torch.FloatTensor(
                (action_space.high - action_space.low) / 2.)
            self.action_bias = torch.FloatTensor(
                (action_space.high + action_space.low) / 2.)
    
    def forward(self, state):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        mean = self.mean_linear(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, LOG_SIG_MIN, LOG_SIG_MAX) # 限制范围
        return mean, log_std
    

    def sample(self, state):
        mean, log_std = self.forward(state)
        std = log_std.exp()
        normal = Normal(mean, std)

        # 利用 forward 产生的 std 和 mean，构建一个高斯分布，并抽取一个原始的虚拟动作 x_t
        x_t = normal.rsample() 
        # x_t 是一个未经过任何变换的动作，可能会超出动作空间的范围(-∞, ∞)，因此需要进行缩放和偏移
        y_t = torch.tanh(x_t)
        action = y_t * self.action_scale + self.action_bias
        # 计算出抽取的原始动作的 log 概率值，并进行调整以考虑 tanh 变换的影响
        log_prob = normal.log_prob(x_t)

        # 当对一个随机变量进行非线性变换时，它的概率密度函数也会发生变换
        # 在这里，tanh 函数将原始动作 x_t 映射到 (-1, 1) 的范围内，这会改变动作的概率分布
        # 因此，\Pi(a) = p(x)|dx/da| = (p(x))/|da/dx|
        # 求对数后，log \Pi(a) = log p(x) - log |da/dx| = log p(x) - log (da/dx)
        # 其中 a = tanh(x) * action_scale + action_bias，da/dx = (1 - tanh(x)^2) * action_scale
        # epsilon 是为了防止数值不稳定而添加的一个小常数，确保在计算 log 时不会出现 log(0) 的情况
        log_prob -= torch.log(self.action_scale * (1-y_t.pow(2)) + epsilon)
        # 得到多个维度的 action 联合概率，联合概率为\Pi(a)相乘，但此处是对数概率，所以是相加
        log_prob = log_prob.sum(1, keepdim=True)
        # mean 是给测试\评估所使用的动作 (分布内mean代表最确定的动作)，经过 tanh 函数进行缩放
        # 并且根据 action_scale 和 action_bias 进行调整，使得输出的动作在动作空间内
        mean = torch.tanh(mean) * self.action_scale + self.action_bias
        
        return action, log_prob, mean # 训练用动作，log_prob 用于计算损失，mean 评估和测试用动作

    def to(self, device):
        self.action_scale = self.action_scale.to(device)
        self.action_bias = self.action_bias.to(device)
        return super(GaussianPolicy, self).to(device)

class DeterministicPolicy(nn.Module):
    # DeterministicPolicy 是 DDPG 算法中使用的策略网络，输出一个确定性的动作，而不是一个概率分布
    def __init__(self, num_inputs, num_actions, hidden_dim, action_space=None):
        super(DeterministicPolicy, self).__init__()
        self.linear1 = nn.Linear(num_inputs, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, hidden_dim)

        self.mean = nn.Linear(hidden_dim, num_actions)
        self.noise = torch.Tensor(num_actions)

        self.apply(weights_init_)

        # action rescaling 动作缩放映射
        if action_space is None:
            self.action_scale = 1.
            self.action_bias = 0.
        else:
            self.action_scale = torch.FloatTensor(
                (action_space.high - action_space.low) / 2.
            )
            self.action_bias = torch.FloatTensor(
                (action_space.high + action_space.low) / 2.
            )

    
    def forward(self, state):
        x = F.relu(self.linear1(state))
        x = F.relu(self.linear2(x))
        # mean 实际上是动作的输出，经过 tanh 函数进行缩放，并且根据 action_scale 和 action_bias 进行调整，使得输出的动作在动作空间内
        mean = torch.tanh(self.mean(x)) * self.action_scale + self.action_bias
        return mean
    

## SAC Algorithm

In [ ]:
class SAC(object):
    def __init__(self, num_inputs, action_space, 
                 gamma, tau, alpha, policy, target_update_interval, automatic_entropy_tuning, 
                 hidden_size, critic_lr, policy_lr, alpha_lr):
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha

        self.policy_type = policy
        self.target_update_interval = target_update_interval
        self.automatic_entropy_tuning = automatic_entropy_tuning

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.critic = QNetwork(num_inputs, action_space.shape[0], hidden_size).to(device=self.device)
        self.critic_optim = Adam(self.critic.parameters(), lr=critic_lr)

        self.critic_target = QNetwork(num_inputs, action_space.shape[0], hidden_size).to(self.device)
        hard_update(self.critic_target, self.critic)

        if self.policy_type == "Gaussian":
            # Target Entropy = −dim(A) (e.g. , -6 for HalfCheetah-v2) as given in the SAC paper
            if self.automatic_entropy_tuning is True:
                # 自动调节 alpha 的值，以使得策略的熵接近目标熵
                # torch.prod 是将所有元素乘起来，得到动作空间维度的乘积，即动作空间的维度数 dim(A)
                # 这是一个启发式规则: 目标熵通常设置为动作空间维度的负值 -dim(A)
                # 这样可以鼓励策略在动作空间中保持足够的随机性 (reward_in_sac = reward + alpha * entropy)，从而促进探索
                # action_space.shape 会返回一个元组 (dim(A),) 表示动作空间的维度数 因此乘出来的结果就是 dim(A)
                self.target_entropy = -torch.prod(torch.Tensor(action_space.shape).to(self.device)).item()
                self.log_alpha = torch.zeros(1, requires_grad=True, device=self.device) # log形式保证 alpha 始终为正数
                self.alpha_optim = Adam([self.log_alpha], lr=alpha_lr)
            
            self.policy = GaussianPolicy(num_inputs, action_space.shape[0], hidden_size, action_space).to(self.device)
            self.policy_optim = Adam(self.policy.parameters(), lr=policy_lr)

        else:
            self.alpha = 0
            self.automatic_entropy_tuning = False
            self.policy = DeterministicPolicy(num_inputs, action_space.shape[0], hidden_size, action_space).to(self.device)
            self.policy_optim = Adam(self.policy.parameters(), lr=policy_lr)

    def select_action(self, state, evaluate=False):
        state = torch.FloatTensor(state).to(self.device).unsqueeze(0)
        if evaluate is False:
            action, _, _ = self.policy.sample(state)
        else:
            _, _, action = self.policy.sample(state)

        return action.detach().cpu().numpy()[0]
    
    def update_parameters(self, memory, batch_size, updates):
        # 从经验回放缓冲区中采样一个批次的转换数据
        state_batch, action_batch, reward_batch, next_state_batch, mask_batch = memory.sample(batch_size=batch_size)

        state_batch = torch.FloatTensor(state_batch).to(self.device)
        next_state_batch = torch.FloatTensor(next_state_batch).to(self.device)
        action_batch = torch.FloatTensor(action_batch).to(self.device)
        reward_batch = torch.FloatTensor(reward_batch).to(self.device).unsqueeze(1)
        mask_batch = torch.FloatTensor(mask_batch).to(self.device).unsqueeze(1)

        with torch.no_grad():
            next_state_action, next_state_log_pi, _ = self.policy.sample(next_state_batch)
            qf1_next_target, qf2_next_target = self.critic_target(next_state_batch, next_state_action)
            # 相当于 + alpha * Entropy，增加了对策略熵的惩罚，鼓励策略保持足够的随机性，从而促进探索
            min_qf_next_target = torch.min(qf1_next_target, qf2_next_target) - self.alpha * next_state_log_pi
            # next_q_value 为TD target，是当前奖励加上折扣后的未来奖励的估计值  
            # mask_batch 是一个二进制掩码，指示下一个状态是否为终止状态，如果是终止状态，则未来奖励为0
            next_q_value = reward_batch + mask_batch * self.gamma * (min_qf_next_target)

        qf1, qf2 = self.critic(state_batch, action_batch)
        qf1_loss = F.mse_loss(qf1, next_q_value)
        qf2_loss = F.mse_loss(qf2, next_q_value)
        qf_loss = qf1_loss + qf2_loss

        self.critic_optim.zero_grad()
        qf_loss.backward()
        self.critic_optim.step()


        pi, log_pi, _ = self.policy.sample(state_batch)
        qf1_pi, qf2_pi = self.critic(state_batch, pi)
        min_qf_pi = torch.min(qf1_pi, qf2_pi)

        # Actor loss 由两部分组成：Q 值和熵项  目标是最大化未来的总收益: Q(s,a) + alpha * Entropy
        # 最大化 min_qf_pi - alpha * log_pi 等价于最小化 alpha * log_pi - min_qf_pi
        # .mean() 是对批次中的所有样本求平均，以得到一个标量损失值，便于反向传播和优化 符合期望 E 的定义
        policy_loss = ((self.alpha * log_pi) - min_qf_pi).mean()
        self.policy_optim.zero_grad()
        policy_loss.backward()
        self.policy_optim.step()

        if self.automatic_entropy_tuning:
            # (log_pi + self.target_entropy).detach() 截断反向传播的梯度流，使得 alpha 的更新不受 log_pi 和 target_entropy 的影响
            # 原本的 loss 公式为 alpha_loss = -E[alpha * (log_pi + target_entropy)] 此处用 log_alpha 是让 Adam 优化器更新更加平滑稳定
            alpha_loss = -(self.log_alpha * (log_pi + self.target_entropy).detach()).mean()

            self.alpha_optim.zero_grad()
            alpha_loss.backward()
            self.alpha_optim.step()

            self.alpha = self.log_alpha.exp()
            alpha_tlogs = self.alpha.clone() # For TensorboardX logs
        else:
            alpha_loss = torch.tensor(0.).to(self.device)
            alpha_tlogs = torch.tensor(self.alpha)
        
        if updates % self.target_update_interval == 0:
            soft_update(self.critic_target, self.critic, self.tau)
        
        return qf1_loss.item(), qf2_loss.item(), policy_loss.item(), alpha_loss.item(), alpha_tlogs.item()



## Replay Memory

In [14]:
class ReplayMemory:
    def __init__(self, capacity, seed):
        random.seed(seed)
        self.capacity = capacity
        self.buffer = []
        self.position = 0
    
    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    
    def __len__(self):
        return len(self.buffer)
    
    def save_buffer(self, env_name, suffix="", save_path=None):
        if not os.path.exists('checkpoints/'):
            os.makedirs('checkpoints/')
        
        if save_path is None:
            save_path = "checkpoints/sac_buffer_{}_{}".format(env_name, suffix)
        print('Saving buffer to {}'.format(save_path))

        with open(save_path, 'wb') as f:
            pickle.dump(self.buffer, f)
    
    def load_buffer(self, save_path):
        print('Loading buffer from {}'.format(save_path))
        with open(save_path, 'rb') as f:
            self.buffer = pickle.load(f)
            self.position = len(self.buffer) % self.capacity

            

## Hyperparameters

In [25]:
env_name = "HalfCheetah-v4"
policy = "Gaussian"
eval = True
gamma = 0.99
tau = 0.005
critic_lr = 3e-4
policy_lr = 3e-4
alpha_lr = 3e-4
alpha = 0.2
automatic_entropy_tuning = True
seed = 42
batch_size = 256
num_steps = 2000
hidden_size = 256
updates_per_step = 1
start_steps = 10000
target_update_interval = 10
replay_size = 1000000
cuda = True


## Main Loop

In [ ]:
# enviroment
env = gym.make(env_name)
env.seed(seed)
env.action_space.seed(seed)

torch.manual_seed(seed)
np.random.seed(seed)

# agent
agent = SAC(num_inputs=env.observation_space.shape[0], action_space = env.action_space,
            gamma=gamma, tau=tau, alpha=alpha, policy=policy, target_update_interval=target_update_interval,
            automatic_entropy_tuning=automatic_entropy_tuning, hidden_size=hidden_size, critic_lr=critic_lr,
            policy_lr=policy_lr, alpha_lr=alpha_lr)

writer = SummaryWriter('runs/{}_SAC_{}_{}_{}'.format(datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S"), env_name, 
                                                                       policy, "autotune" if automatic_entropy_tuning else ""))

# Memory
memory = ReplayMemory(replay_size, seed)

total_numsteps = 0
updates = 0

for i_episode in itertools.count(1):
    episode_reward = 0
    episode_steps = 0
    done = False
    state = env.reset()

    while not done:
        if start_steps > total_numsteps:
            action = env.action_space.sample()
        else:
            action = agent.select_action(state)
        
        if len(memory) > batch_size:
            for i in range(updates_per_step):
                critic_1_loss, critic_2_loss, policy_loss, ent_loss, alpha = agent.update_parameters(memory, batch_size, updates)

                writer.add_scalar('loss/critic_1', critic_1_loss, updates)
                writer.add_scalar('loss/critic_2', critic_2_loss, updates)
                writer.add_scalar('loss/policy', policy_loss, updates)
                writer.add_scalar('loss/entropy_loss', ent_loss, updates)
                writer.add_scalar('entropy_temprature/alpha', alpha, updates)
                updates += 1
        
        next_state, reward, done, _ = env.step(action)
        episode_steps += 1
        total_numsteps += 1
        episode_reward += reward

        mask = 1 if episode_steps == env._max_episode_steps else float(not done)
        memory.push(state, action, reward, next_state, mask)

        state = next_state
    
    if total_numsteps > num_steps:
        break

    writer.add_scalar('reward/train', episode_reward, i_episode)
    print("Episode: {}, total numsteps: {}, episode steps: {}, reward: {}".format(i_episode, total_numsteps, episode_steps, round(episode_reward, 2)))

    if i_episode % 10 == 0 and eval is True:
        avg_reward = 0.
        episodes = 10
        for _ in range(episodes):
            state = env.reset()
            episode_reward = 0
            done = False
            while not done:
                action = agent.select_action(state, evaluate=True)

                next_state, reward, done, _ = env.step(action)
                episode_reward += reward

                state = next_state
            
            avg_reward += episode_reward
        avg_reward /= episodes


        writer.add_scalar('avg_reward/test', avg_reward, i_episode)

        print("--------------------------------------")
        print("Test Episodes: {}, Avg. Reward: {}".format(episodes, round(avg_reward, 2)))
        print("--------------------------------------")
        


## Render the trained agent

In [ ]:
test_state = env.reset()
for i in range(5000):
    test_action = agent.select_action(test_state)
    test_state, _, _ , _ = env.step(test_action)
    env.render()



In [ ]:
env.close()